<a href="https://colab.research.google.com/github/karkessler/dhbw-mathe3/blob/main/notebooks/gradient_descent_rosenbrock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradientenabstiegsverfahren als numerisches Optimierungsverfahren

Dieses Notebook behandelt den Gradientenabstieg so, wie er in der numerischen Mathematik
ueblicherweise eingefuehrt wird: als iteratives Verfahren zur unrestringierten Minimierung
einer differenzierbaren Funktion $f: \mathbb{R}^n \to \mathbb{R}$.

Testfunktion ist die **Rosenbrock-Funktion** (auch "Bananenfunktion" genannt), ein Standardbeispiel
in der nichtlinearen Optimierung:

$$f(x, y) = (a - x)^2 + b \cdot (y - x^2)^2, \qquad \text{ueblich } a=1,\ b=100$$

Sie hat ein einziges globales Minimum bei $(a, a^2) = (1, 1)$, liegt aber in einem sehr flachen,
gekruemmten Tal -- ein gutes Beispiel dafuer, warum reiner Gradientenabstieg dort nur langsam
konvergiert und wie stark die Wahl der Schrittweite (Lernrate) $\eta$ das Ergebnis beeinflusst.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def rosenbrock(x, y, a=1.0, b=100.0):
    """Rosenbrock-Funktion ("Bananenfunktion"): Standardtestfunktion
    fuer nichtlineare Optimierungsverfahren. Globales Minimum bei (a, a^2)."""
    return (a - x) ** 2 + b * (y - x ** 2) ** 2


def rosenbrock_grad(x, y, a=1.0, b=100.0):
    dfdx = -2 * (a - x) - 4 * b * x * (y - x ** 2)
    dfdy = 2 * b * (y - x ** 2)
    return np.array([dfdx, dfdy])


def gradient_descent(grad_f, x0, eta, tol=1e-6, max_iter=10000):
    """
    Allgemeines Gradientenabstiegsverfahren fuer f: R^n -> R.

    Parameter
    ---------
    grad_f   : Funktion, die den Gradienten an der Stelle x zurueckgibt
    x0       : Startpunkt (numpy array)
    eta      : Schrittweite (Lernrate)
    tol      : Abbruchtoleranz fuer die Gradientennorm
    max_iter : maximale Anzahl Iterationen

    Rueckgabe
    ---------
    x        : gefundener Punkt
    path     : Liste aller besuchten Punkte (fuer die Visualisierung)
    """
    x = np.array(x0, dtype=float)
    path = [x.copy()]

    for k in range(max_iter):
        g = grad_f(*x)
        if np.linalg.norm(g) < tol:
            break
        x = x - eta * g
        path.append(x.copy())

    return x, np.array(path)

## Verhalten bei unterschiedlichen Lernraten

In [ ]:
x0 = np.array([-1.2, 1.0])

print("Gradientenabstieg auf der Rosenbrock-Funktion")
print(f"Startpunkt: {x0}, gesuchtes Minimum: (1, 1)\n")

for eta in [0.0001, 0.001, 0.002]:
    x_opt, path = gradient_descent(rosenbrock_grad, x0, eta, max_iter=20000)
    f_opt = rosenbrock(*x_opt)
    print(f"eta = {eta:<8} -> {len(path)-1:5d} Schritte, "
          f"x = ({x_opt[0]:.4f}, {x_opt[1]:.4f}), f(x) = {f_opt:.6f}")

## Konvergenzverlauf und Weg im Hoehenlinienbild

In [ ]:
eta = 0.001
x_opt, path = gradient_descent(rosenbrock_grad, x0, eta, max_iter=20000)
f_values = [rosenbrock(*p) for p in path]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Konvergenzkurve (log-Skala)
axes[0].plot(f_values)
axes[0].set_yscale("log")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("f(x)  (log-Skala)")
axes[0].set_title(f"Konvergenz, eta = {eta}")

# Hoehenlinien + Weg des Verfahrens
xs = np.linspace(-1.5, 1.5, 400)
ys = np.linspace(-0.5, 1.5, 400)
X, Y = np.meshgrid(xs, ys)
Z = rosenbrock(X, Y)

axes[1].contour(X, Y, Z, levels=np.logspace(-1, 3.5, 20), cmap="viridis")
axes[1].plot(path[:, 0], path[:, 1], "r.-", markersize=3, linewidth=0.8,
             label="Weg des Gradientenabstiegs")
axes[1].plot(1, 1, "k*", markersize=12, label="Minimum (1,1)")
axes[1].plot(*x0, "go", label="Start")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Rosenbrock-Funktion: Hoehenlinien und Abstiegsweg")
axes[1].legend(loc="upper left", fontsize=8)

fig.tight_layout()
plt.show()

print("\nZum Vergleich: Das Newton-Verfahren nutzt zusaetzlich die Kruemmung")
print("(Hesse-Matrix) und konvergiert auf dieser Funktion in wenigen Schritten")
print("statt Tausenden -- Thema eines spaeteren Kapitels dieser Vorlesung.")